# Online Course Completion Events — Bronze / Silver / Gold Pipeline

Follows the same pattern as `700-wide-katas.ipynb`:

`raw CSV -> bronze/ (land as-is) -> silver/ (clean: remove nulls,
standardize types, deduplicate) -> gold/ (aggregate to business
metrics) -> DQ checks (nulls, ranges, grain, coverage) -> serve
(plotly charts)`

Built with DuckDB throughout. Every layer prints a row-count
verification before moving to the next.


## Step 1 — Generate the raw dataset (bronze/course_events_raw.csv)

500 rows of online course completion events with realistic mess:
- `event_id` (`EVT-XXXXX`) — 2% duplicates
- `student_id` — integer 1000-9999
- `event_date` — mixed formats: `2024-MM-DD`, `MM/DD/YYYY`, `Month DD YYYY`
- `course_category` — Data, Engineering, Design, Business, Security
- `completion_pct` — float 0-100, 4% nulls
- `time_spent_minutes` — integer 10-480
- `status` — completed / in_progress / dropped (70/20/10 split)

In [1]:
import random
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta

# ============================================================
# Set seeds for full reproducibility
# ============================================================
random.seed(42)
np.random.seed(42)

# ============================================================
# Configuration constants
# ============================================================
NUM_ROWS = 500
DUPLICATE_RATE = 0.02        # 2% duplicate event_ids
NULL_COMPLETION_RATE = 0.04  # 4% null completion_pct

COURSE_CATEGORIES = ["Data", "Engineering", "Design", "Business", "Security"]
STATUSES = ["completed", "in_progress", "dropped"]
STATUS_WEIGHTS = [0.70, 0.20, 0.10]

DATE_START = datetime(2024, 1, 1)
DATE_END = datetime(2024, 12, 31)
DATE_RANGE_DAYS = (DATE_END - DATE_START).days

# ============================================================
# Helper: generate a random date within 2024
# ============================================================
def random_date():
    offset = random.randint(0, DATE_RANGE_DAYS)
    return DATE_START + timedelta(days=offset)

# ============================================================
# Helper: format a date in one of three random formats
#   "2024-01-15"     — YYYY-MM-DD
#   "01/15/2024"     — MM/DD/YYYY
#   "January 15 2024" — Month DD YYYY
# ============================================================
def format_date(dt):
    fmt = random.choice(["iso", "mdy", "month_text"])
    if fmt == "iso":
        return dt.strftime("%Y-%m-%d")
    elif fmt == "mdy":
        return dt.strftime("%m/%d/%Y")
    else:
        return dt.strftime("%B %d %Y")

# ============================================================
# STEP 1: Generate unique base event_ids, then introduce dupes
# ============================================================
num_duplicates = int(NUM_ROWS * DUPLICATE_RATE)   # 10 duplicates
num_unique = NUM_ROWS - num_duplicates             # 490 unique ids

unique_ids = random.sample(range(10000, 99999), num_unique)
event_ids = [f"EVT-{uid}" for uid in unique_ids]

duplicates = random.choices(event_ids, k=num_duplicates)
event_ids.extend(duplicates)

random.shuffle(event_ids)

# ============================================================
# STEP 2: Generate the remaining columns
# ============================================================
student_ids = [random.randint(1000, 9999) for _ in range(NUM_ROWS)]
event_dates = [format_date(random_date()) for _ in range(NUM_ROWS)]
categories = [random.choice(COURSE_CATEGORIES) for _ in range(NUM_ROWS)]
time_spent = [random.randint(10, 480) for _ in range(NUM_ROWS)]
statuses = random.choices(STATUSES, weights=STATUS_WEIGHTS, k=NUM_ROWS)

# ============================================================
# STEP 3: Generate completion_pct with 4% nulls
# ============================================================
completion_pcts = []
for i in range(NUM_ROWS):
    if random.random() < NULL_COMPLETION_RATE:
        completion_pcts.append(np.nan)
    else:
        completion_pcts.append(round(random.uniform(0, 100), 2))

# ============================================================
# STEP 4: Assemble into a DataFrame
# ============================================================
df = pd.DataFrame({
    "event_id":            event_ids,
    "student_id":          student_ids,
    "event_date":          event_dates,
    "course_category":     categories,
    "completion_pct":      completion_pcts,
    "time_spent_minutes":  time_spent,
    "status":              statuses,
})

# ============================================================
# STEP 5: Save to bronze/course_events_raw.csv
# ============================================================
output_dir = "bronze"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "course_events_raw.csv")
df.to_csv(output_path, index=False)

# ============================================================
# STEP 6: Print summary statistics + row-count verification
# ============================================================
total_rows = len(df)
null_count = df["completion_pct"].isna().sum()
duplicate_count = df["event_id"].duplicated(keep=False).sum()

def detect_format(date_str):
    if "-" in date_str and date_str[:4].isdigit():
        return "YYYY-MM-DD"
    elif "/" in date_str:
        return "MM/DD/YYYY"
    else:
        return "Month DD YYYY"

unique_formats = sorted(set(detect_format(d) for d in df["event_date"]))

print(f"Total rows              : {total_rows}")
print(f"Null completion_pct     : {null_count} ({null_count/total_rows*100:.1f}%)")
print(f"Duplicate event_ids     : {duplicate_count} rows ({df['event_id'].duplicated().sum()} non-first occurrences)")
print(f"Unique date formats     : {len(unique_formats)} -- {unique_formats}")
print(f"Status distribution     : {df['status'].value_counts(normalize=True).round(3).to_dict()}")
print(f"\nSaved to: {output_path}")
print(f"\n--- BRONZE ROW-COUNT VERIFICATION ---")
print(f"Expected rows : {NUM_ROWS}")
print(f"Actual rows   : {total_rows}")
print(f"Match         : {'OK' if total_rows == NUM_ROWS else 'FAIL'}")
print(f"\n--- First 5 rows ---")
print(df.head().to_string(index=False))


Total rows              : 500
Null completion_pct     : 22 (4.4%)
Duplicate event_ids     : 20 rows (10 non-first occurrences)
Unique date formats     : 3 -- ['MM/DD/YYYY', 'Month DD YYYY', 'YYYY-MM-DD']
Status distribution     : {'completed': 0.684, 'in_progress': 0.2, 'dropped': 0.116}

Saved to: bronze/course_events_raw.csv

--- BRONZE ROW-COUNT VERIFICATION ---
Expected rows : 500
Actual rows   : 500
Match         : OK

--- First 5 rows ---
 event_id  student_id        event_date course_category  completion_pct  time_spent_minutes    status
EVT-26828        8679        2024-08-08        Security           15.71                 257 completed
EVT-18981        5982        03/07/2024     Engineering           20.56                 410 completed
EVT-79163        7594 September 29 2024     Engineering           69.72                  49 completed
EVT-96474        5460     April 27 2024        Security           73.00                 194 completed
EVT-97684        9199   October 13 2024  

## Step 2 — Bronze -> Silver (clean, standardize, deduplicate)

- Drop rows where `completion_pct IS NULL`
- Standardize `event_date` to `DATE` via `TRY_STRPTIME` across all three known formats
- Deduplicate `event_id`, keeping the row with the highest `completion_pct` (tie-broken by `time_spent_minutes`)
- Write to `silver/course_events_clean.parquet`

In [2]:
import duckdb
import os

con = duckdb.connect(database=":memory:")

BRONZE_CSV     = "bronze/course_events_raw.csv"
SILVER_DIR     = "silver"
SILVER_PARQUET = os.path.join(SILVER_DIR, "course_events_clean.parquet")

os.makedirs(SILVER_DIR, exist_ok=True)

# ============================================================
# STEP 0: Bronze-layer snapshot (before cleaning)
# ============================================================
before = con.execute(f"""
    SELECT
        COUNT(*)                                                   AS total_rows,
        SUM(CASE WHEN completion_pct IS NULL THEN 1 ELSE 0 END)   AS null_completion_pct
    FROM read_csv_auto('{BRONZE_CSV}')
""").fetchone()

print("=== BRONZE (before cleaning) ===")
print(f"  Total rows           : {before[0]}")
print(f"  Null completion_pct  : {before[1]}")

# ============================================================
# STEP 1-4: Read, drop nulls, standardize dates, deduplicate
#
#   Layer 1 (raw_with_dates):
#       - Read CSV with event_date forced to VARCHAR
#       - Drop rows where completion_pct IS NULL
#       - Standardize event_date to DATE via COALESCE + TRY_STRPTIME
#         across all three known formats
#
#   Layer 2 (deduped):
#       - QUALIFY with ROW_NUMBER to keep one row per event_id,
#         preferring the highest completion_pct (most complete
#         signal), tie-broken by highest time_spent_minutes
# ============================================================
clean_query = f"""
    WITH raw_with_dates AS (
        SELECT
            event_id,
            student_id,
            COALESCE(
                TRY_STRPTIME(event_date, '%Y-%m-%d'),   -- 2024-01-15
                TRY_STRPTIME(event_date, '%m/%d/%Y'),   -- 01/15/2024
                TRY_STRPTIME(event_date, '%B %d %Y')    -- January 15 2024
            )::DATE AS event_date,
            course_category,
            completion_pct::DOUBLE AS completion_pct,
            time_spent_minutes::INTEGER AS time_spent_minutes,
            status
        FROM read_csv_auto('{BRONZE_CSV}', types={{'event_date': 'VARCHAR'}})
        WHERE completion_pct IS NOT NULL
    ),

    deduped AS (
        SELECT *
        FROM   raw_with_dates
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY event_id
            ORDER BY     completion_pct DESC, time_spent_minutes DESC
        ) = 1
    )

    SELECT * FROM deduped
    ORDER BY event_date, event_id
"""

preview_df = con.execute(clean_query).fetchdf()
print(f"\n=== CLEANING RESULT ===")
print(f"  Rows after cleaning : {len(preview_df)}")
print(f"\n  First 5 rows:")
print(preview_df.head().to_string(index=False))

# ============================================================
# STEP 5: Write to silver/course_events_clean.parquet
# ============================================================
con.execute(f"""
    COPY (
        {clean_query}
    ) TO '{SILVER_PARQUET}' (FORMAT PARQUET)
""")

print(f"\n  Parquet written to  : {SILVER_PARQUET}")

# ============================================================
# STEP 6: Row-count verification on the silver parquet file
# ============================================================
verify = con.execute(f"""
    SELECT
        COUNT(*)                                                  AS silver_row_count,
        SUM(CASE WHEN completion_pct IS NULL THEN 1 ELSE 0 END)  AS null_completion_count,
        SUM(CASE WHEN event_date     IS NULL THEN 1 ELSE 0 END)  AS null_date_count
    FROM '{SILVER_PARQUET}'
""").fetchone()

dup_check = con.execute(f"""
    SELECT COUNT(*) AS duplicate_event_ids
    FROM (
        SELECT   event_id
        FROM     '{SILVER_PARQUET}'
        GROUP BY event_id
        HAVING   COUNT(*) > 1
    )
""").fetchone()[0]

date_type_check = con.execute(f"""
    SELECT typeof(event_date) AS date_type
    FROM   '{SILVER_PARQUET}'
    LIMIT  1
""").fetchone()[0]

print(f"\n=== SILVER ROW-COUNT VERIFICATION ===")
print(f"  Row count             : {verify[0]}")
print(f"  Null completion_pct   : {verify[1]}  {'OK' if verify[1] == 0 else 'FAIL'}")
print(f"  Null event_date       : {verify[2]}  {'OK' if verify[2] == 0 else 'FAIL'}")
print(f"  Duplicate event_ids   : {dup_check}  {'OK' if dup_check == 0 else 'FAIL'}")
print(f"  event_date type       : {date_type_check}")

rows_removed = before[0] - verify[0]
print(f"\n=== SUMMARY ===")
print(f"  Bronze rows  : {before[0]}")
print(f"  Silver rows  : {verify[0]}")
print(f"  Rows removed : {rows_removed} (nulls + duplicate collapses)")
print(f"\n[OK] Bronze -> Silver pipeline complete.")


=== BRONZE (before cleaning) ===
  Total rows           : 500
  Null completion_pct  : 22

=== CLEANING RESULT ===
  Rows after cleaning : 468

  First 5 rows:
 event_id  student_id event_date course_category  completion_pct  time_spent_minutes    status
EVT-67422        8957 2024-01-03            Data           23.03                 241   dropped
EVT-44970        8847 2024-01-05        Business           53.80                 147 completed
EVT-86099        4612 2024-01-05        Business           17.06                 259   dropped
EVT-60140        5183 2024-01-09        Security           13.72                 241 completed
EVT-23396        5634 2024-01-10            Data           34.87                 308 completed

  Parquet written to  : silver/course_events_clean.parquet

=== SILVER ROW-COUNT VERIFICATION ===
  Row count             : 468
  Null completion_pct   : 0  OK
  Null event_date       : 0  OK
  Duplicate event_ids   : 0  OK
  event_date type       : DATE

=== SUMMARY =

## Step 3 — Silver -> Gold (aggregate to business metrics)

**`gold/daily_completions_by_category`** — grain: `(event_date, course_category)`, scoped to `status = 'completed'`
- `avg_completion_pct` = AVG(completion_pct)
- `completion_count` = COUNT(DISTINCT event_id)

**`gold/dropout_rate`** — grain: `event_date`, all statuses
- `total_enrollments` = COUNT(DISTINCT event_id) (completed + in_progress + dropped)
- `dropped_count` = COUNT(DISTINCT event_id) WHERE status='dropped'
- `dropout_rate_pct` = dropped_count / total_enrollments * 100

In [3]:
import duckdb
import os

con = duckdb.connect(database=":memory:")

SILVER_PARQUET = "silver/course_events_clean.parquet"
GOLD_DIR       = "gold"
GOLD_DAILY     = os.path.join(GOLD_DIR, "daily_completions_by_category.parquet")
GOLD_DROPOUT   = os.path.join(GOLD_DIR, "dropout_rate.parquet")

os.makedirs(GOLD_DIR, exist_ok=True)

# ============================================================
# Quick look at the silver source
# ============================================================
src = con.execute(f"""
    SELECT
        COUNT(*)                    AS total_rows,
        COUNT(DISTINCT event_id)    AS unique_events,
        MIN(event_date)             AS min_date,
        MAX(event_date)             AS max_date,
        COUNT(DISTINCT course_category) AS categories
    FROM '{SILVER_PARQUET}'
""").fetchone()

print("=== SILVER SOURCE ===")
print(f"  Rows           : {src[0]}")
print(f"  Unique events  : {src[1]}")
print(f"  Date range     : {src[2]} to {src[3]}")
print(f"  Categories     : {src[4]}")

# ============================================================
# GOLD TABLE 1: daily_completions_by_category
#
# Grain : one row per (event_date, course_category)
# Scope : only 'completed' events contribute
#
# Logic :
#   - Filter to status = 'completed'
#   - GROUP BY the two grain columns
#   - AVG(completion_pct) as avg_completion_pct
#   - COUNT(DISTINCT event_id) as completion_count
# ============================================================
daily_query = f"""
    SELECT
        event_date,
        course_category,
        ROUND(AVG(completion_pct), 2)  AS avg_completion_pct,
        COUNT(DISTINCT event_id)       AS completion_count
    FROM   '{SILVER_PARQUET}'
    WHERE  status = 'completed'
    GROUP  BY event_date, course_category
    ORDER  BY event_date, course_category
"""

con.execute(f"COPY ({daily_query}) TO '{GOLD_DAILY}' (FORMAT PARQUET)")
print(f"\n[OK] Written: {GOLD_DAILY}")

# ============================================================
# GOLD TABLE 2: dropout_rate
#
# Grain : one row per event_date
# Scope : all statuses (completed + in_progress + dropped)
#
# Logic :
#   - total_enrollments = COUNT(DISTINCT event_id) across all statuses
#   - dropped_count      = COUNT(DISTINCT event_id) WHERE status='dropped'
#   - dropout_rate_pct   = dropped / total * 100, rounded to 2dp
#
#   NULLIF guards against division by zero on dates with zero
#   qualifying events (shouldn't happen, but safe).
# ============================================================
dropout_query = f"""
    SELECT
        event_date,
        COUNT(DISTINCT event_id) AS total_enrollments,
        COUNT(DISTINCT CASE
            WHEN status = 'dropped' THEN event_id
        END) AS dropped_count,
        ROUND(
            COUNT(DISTINCT CASE
                WHEN status = 'dropped' THEN event_id
            END) * 100.0
            / NULLIF(COUNT(DISTINCT event_id), 0),
            2
        ) AS dropout_rate_pct
    FROM   '{SILVER_PARQUET}'
    GROUP  BY event_date
    ORDER  BY event_date
"""

con.execute(f"COPY ({dropout_query}) TO '{GOLD_DROPOUT}' (FORMAT PARQUET)")
print(f"[OK] Written: {GOLD_DROPOUT}")

# ============================================================
# ROW-COUNT VERIFICATION 1: daily_completions_by_category
# ============================================================
grain_check = con.execute(f"""
    SELECT COUNT(*) AS duplicates
    FROM (
        SELECT   event_date, course_category
        FROM     '{GOLD_DAILY}'
        GROUP BY event_date, course_category
        HAVING   COUNT(*) > 1
    )
""").fetchone()[0]

daily_rows = con.execute(f"SELECT COUNT(*) FROM '{GOLD_DAILY}'").fetchone()[0]

daily_stats = con.execute(f"""
    SELECT
        MIN(avg_completion_pct)  AS min_pct,
        MAX(avg_completion_pct)  AS max_pct,
        SUM(completion_count)    AS grand_total_completions
    FROM '{GOLD_DAILY}'
""").fetchone()

print(f"\n=== GOLD TABLE 1: daily_completions_by_category ===")
print(f"  Row count               : {daily_rows}")
print(f"  Grain duplicates        : {grain_check}  {'OK' if grain_check == 0 else 'FAIL'}")
print(f"  avg_completion_pct range: {daily_stats[0]} to {daily_stats[1]}")
print(f"  Grand total completions : {daily_stats[2]}")

# ============================================================
# ROW-COUNT VERIFICATION 2: dropout_rate
# ============================================================
dropout_rows = con.execute(f"SELECT COUNT(*) FROM '{GOLD_DROPOUT}'").fetchone()[0]

date_grain_check = con.execute(f"""
    SELECT COUNT(*) AS duplicates
    FROM (
        SELECT   event_date
        FROM     '{GOLD_DROPOUT}'
        GROUP BY event_date
        HAVING   COUNT(*) > 1
    )
""").fetchone()[0]

rate_bounds = con.execute(f"""
    SELECT
        MIN(dropout_rate_pct)  AS min_rate,
        MAX(dropout_rate_pct)  AS max_rate,
        AVG(dropout_rate_pct)  AS avg_rate
    FROM '{GOLD_DROPOUT}'
""").fetchone()

rate_violations = con.execute(f"""
    SELECT COUNT(*) AS out_of_range
    FROM   '{GOLD_DROPOUT}'
    WHERE  dropout_rate_pct < 0 OR dropout_rate_pct > 100
""").fetchone()[0]

print(f"\n=== GOLD TABLE 2: dropout_rate ===")
print(f"  Row count              : {dropout_rows}")
print(f"  Date grain duplicates  : {date_grain_check}  {'OK' if date_grain_check == 0 else 'FAIL'}")
print(f"  Rate range             : {rate_bounds[0]}% to {rate_bounds[1]}%")
print(f"  Average dropout rate   : {rate_bounds[2]:.2f}%")
print(f"  Out-of-range rates     : {rate_violations}  {'OK' if rate_violations == 0 else 'FAIL'}")

print(f"\n--- daily_completions_by_category (first 5 rows) ---")
print(con.execute(f"SELECT * FROM '{GOLD_DAILY}' LIMIT 5").fetchdf().to_string(index=False))

print(f"\n--- dropout_rate (first 5 rows) ---")
print(con.execute(f"SELECT * FROM '{GOLD_DROPOUT}' LIMIT 5").fetchdf().to_string(index=False))

print(f"\n=== PIPELINE SUMMARY ===")
print(f"  Silver rows                          : {src[0]}")
print(f"  Gold daily_completions_by_category    : {daily_rows} rows")
print(f"  Gold dropout_rate                     : {dropout_rows} rows")
print(f"  All grain checks passed               : {'YES' if grain_check == 0 and date_grain_check == 0 else 'NO'}")
print(f"  All rate bounds valid                 : {'YES' if rate_violations == 0 else 'NO'}")
print(f"\n[OK] Silver -> Gold pipeline complete.")


=== SILVER SOURCE ===
  Rows           : 468
  Unique events  : 468
  Date range     : 2024-01-03 to 2024-12-31
  Categories     : 5

[OK] Written: gold/daily_completions_by_category.parquet
[OK] Written: gold/dropout_rate.parquet

=== GOLD TABLE 1: daily_completions_by_category ===
  Row count               : 290
  Grain duplicates        : 0  OK
  avg_completion_pct range: 1.86 to 99.73
  Grand total completions : 326

=== GOLD TABLE 2: dropout_rate ===
  Row count              : 260
  Date grain duplicates  : 0  OK
  Rate range             : 0.0% to 100.0%
  Average dropout rate   : 10.79%
  Out-of-range rates     : 0  OK

--- daily_completions_by_category (first 5 rows) ---
event_date course_category  avg_completion_pct  completion_count
2024-01-05        Business               53.80                 1
2024-01-09        Security               13.72                 1
2024-01-10            Data               32.16                 2
2024-01-11     Engineering               54.41       

## Step 4 — Data Quality Checks (6 checks: nulls, ranges, grain, rate bounds)

In [4]:
import duckdb

con = duckdb.connect(database=":memory:")

GOLD_DAILY   = "gold/daily_completions_by_category.parquet"
GOLD_DROPOUT = "gold/dropout_rate.parquet"


# ============================================================
# Helper: run a check and print PASS / FAIL
#
#   rule_name  : human-readable label
#   fail_query : SQL that returns ONLY the failing rows
#
#   Logic:
#     - Execute the fail_query
#     - Zero rows back -> PASS
#     - Any rows back  -> FAIL, show count + up to 3 examples
#
#   Returns True on PASS, False on FAIL.
# ============================================================
def run_check(rule_name, fail_query):
    fail_df = con.execute(fail_query).fetchdf()
    fail_count = len(fail_df)

    if fail_count == 0:
        print(f"  PASS  |  {rule_name}")
        return True
    else:
        print(f"  FAIL  |  {rule_name}")
        print(f"         |  Failing rows: {fail_count}")
        print(f"         |  Examples (up to 3):")
        sample = fail_df.head(3).to_string(index=False)
        for line in sample.split("\n"):
            print(f"         |    {line}")
        return False


# ============================================================
# CHECK 1 (nulls) — daily_completions_by_category:
#   no null event_date / course_category / avg_completion_pct
# ============================================================
def check_1_daily_no_nulls():
    return run_check(
        "Daily: no null event_date / course_category / avg_completion_pct",
        f"""
        SELECT event_date, course_category, avg_completion_pct, completion_count
        FROM   '{GOLD_DAILY}'
        WHERE  event_date         IS NULL
           OR  course_category    IS NULL
           OR  avg_completion_pct IS NULL
        """,
    )


# ============================================================
# CHECK 2 (ranges) — daily_completions_by_category:
#   avg_completion_pct in [0, 100] AND completion_count > 0
# ============================================================
def check_2_daily_ranges():
    return run_check(
        "Daily: avg_completion_pct in [0, 100] and completion_count > 0",
        f"""
        SELECT event_date, course_category, avg_completion_pct, completion_count
        FROM   '{GOLD_DAILY}'
        WHERE  avg_completion_pct < 0.0
           OR  avg_completion_pct > 100.0
           OR  completion_count <= 0
        """,
    )


# ============================================================
# CHECK 3 (grain) — daily_completions_by_category:
#   no duplicate (event_date, course_category)
# ============================================================
def check_3_daily_unique_grain():
    return run_check(
        "Daily: unique grain (event_date, course_category)",
        f"""
        SELECT   event_date, course_category, COUNT(*) AS row_count
        FROM     '{GOLD_DAILY}'
        GROUP BY event_date, course_category
        HAVING   COUNT(*) > 1
        """,
    )


# ============================================================
# CHECK 4 (nulls) — dropout_rate: no null event_date
# ============================================================
def check_4_dropout_no_null_date():
    return run_check(
        "Dropout: no null event_date",
        f"""
        SELECT *
        FROM   '{GOLD_DROPOUT}'
        WHERE  event_date IS NULL
        """,
    )


# ============================================================
# CHECK 5 (grain) — dropout_rate: no duplicate event_date
# ============================================================
def check_5_dropout_unique_grain():
    return run_check(
        "Dropout: unique grain (event_date)",
        f"""
        SELECT   event_date, COUNT(*) AS row_count
        FROM     '{GOLD_DROPOUT}'
        GROUP BY event_date
        HAVING   COUNT(*) > 1
        """,
    )


# ============================================================
# CHECK 6 (rate bounds) — dropout_rate:
#   dropout_rate_pct in [0, 100] AND dropped_count <= total_enrollments
# ============================================================
def check_6_dropout_rate_bounds():
    return run_check(
        "Dropout: dropout_rate_pct in [0, 100] and dropped_count <= total_enrollments",
        f"""
        SELECT event_date, total_enrollments, dropped_count, dropout_rate_pct
        FROM   '{GOLD_DROPOUT}'
        WHERE  dropout_rate_pct < 0.0
           OR  dropout_rate_pct > 100.0
           OR  dropped_count > total_enrollments
        """,
    )


# ============================================================
# RUNNER: execute all 6 checks and print summary
# ============================================================
def run_all_checks():
    print("=" * 62)
    print("  GOLD LAYER DATA QUALITY CHECKS")
    print("=" * 62)

    print(f"\n  daily_completions_by_category ({GOLD_DAILY})")
    print("-" * 62)
    results_daily = [
        check_1_daily_no_nulls(),
        check_2_daily_ranges(),
        check_3_daily_unique_grain(),
    ]

    print(f"\n  dropout_rate ({GOLD_DROPOUT})")
    print("-" * 62)
    results_dropout = [
        check_4_dropout_no_null_date(),
        check_5_dropout_unique_grain(),
        check_6_dropout_rate_bounds(),
    ]

    all_results = results_daily + results_dropout
    passed = sum(all_results)
    total = len(all_results)

    print("\n" + "=" * 62)
    check_names = [
        "1-Nulls", "2-Ranges", "3-Grain",
        "4-Nulls", "5-Grain", "6-RateBounds",
    ]
    if passed == total:
        print(f"  RESULT: {passed}/{total} checks passed  --  ALL CLEAR")
    else:
        failed_names = [n for n, r in zip(check_names, all_results) if not r]
        print(f"  RESULT: {passed}/{total} checks passed")
        print(f"  FAILED: {', '.join(failed_names)}")
    print("=" * 62)

    return passed, total


if __name__ == "__main__":
    run_all_checks()


  GOLD LAYER DATA QUALITY CHECKS

  daily_completions_by_category (gold/daily_completions_by_category.parquet)
--------------------------------------------------------------
  PASS  |  Daily: no null event_date / course_category / avg_completion_pct
  PASS  |  Daily: avg_completion_pct in [0, 100] and completion_count > 0
  PASS  |  Daily: unique grain (event_date, course_category)

  dropout_rate (gold/dropout_rate.parquet)
--------------------------------------------------------------
  PASS  |  Dropout: no null event_date


  PASS  |  Dropout: unique grain (event_date)
  PASS  |  Dropout: dropout_rate_pct in [0, 100] and dropped_count <= total_enrollments

  RESULT: 6/6 checks passed  --  ALL CLEAR


## Step 5 — Serve: Plotly charts

1. Daily average completion % by course category (line, colored by category)
2. Dropout rate over time (line)

In [5]:
import pandas as pd
import plotly.express as px

# ============================================================
# LOAD GOLD TABLES
# ============================================================
daily_df   = pd.read_parquet("gold/daily_completions_by_category.parquet")
dropout_df = pd.read_parquet("gold/dropout_rate.parquet")

daily_df["event_date"]   = pd.to_datetime(daily_df["event_date"])
dropout_df["event_date"] = pd.to_datetime(dropout_df["event_date"])

print(f"daily_completions_by_category rows : {len(daily_df)}")
print(f"dropout_rate rows                  : {len(dropout_df)}")

daily_completions_by_category rows : 290
dropout_rate rows                  : 260


In [6]:
# ============================================================
# CHART 1 — Line: avg_completion_pct over time, colored by course_category
# ============================================================
fig1 = px.line(
    daily_df.sort_values("event_date"),
    x="event_date",
    y="avg_completion_pct",
    color="course_category",
    markers=True,
    title="Daily Average Completion % by Course Category",
    labels={
        "event_date": "Date",
        "avg_completion_pct": "Avg Completion (%)",
        "course_category": "Category",
    },
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig1.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(range=[0, 100]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=40, r=20, t=60, b=40),
    height=420,
)
fig1.show()

In [7]:
# ============================================================
# CHART 2 — Line: dropout_rate_pct over time
# ============================================================
fig2 = px.line(
    dropout_df.sort_values("event_date"),
    x="event_date",
    y="dropout_rate_pct",
    markers=True,
    title="Dropout Rate Over Time",
    labels={
        "event_date": "Date",
        "dropout_rate_pct": "Dropout Rate (%)",
    },
)
fig2.update_traces(
    line=dict(color="#e05252", width=2.5),
    marker=dict(size=5, color="#e05252"),
)
fig2.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(range=[0, max(dropout_df["dropout_rate_pct"].max() * 1.15, 10)]),
    margin=dict(l=40, r=20, t=60, b=40),
    height=380,
)
fig2.show()